# prob_to_sink == 1 fractions (LWCC)

This notebook computes the fraction of nodes with `prob_to_sink == 1` for:
1. all nodes in the LWCC
2. nodes with `in_strength > 0`
3. nodes with `in_strength > out_strength`

The calculation follows the same definition used in the pipeline:
`prob_to_sink = (in_strength - out_strength) / in_strength` (undefined if `in_strength == 0`).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    yaml = None

def _load_config(path: Path) -> dict:
    txt = path.read_text(encoding='utf-8')
    if yaml is not None:
        return yaml.safe_load(txt)
    return json.loads(txt)

def _resolve(project_root: Path, value: str) -> Path:
    p = Path(value).expanduser()
    return p if p.is_absolute() else (project_root / p)

def _detect_cols(columns):
    colset = set(columns)
    preferred = [
        ('source', 'target', 'weight'),
        ('source_id_str', 'target_id_str', 'weight'),
        ('user_id_str', 'retweet_author_id', 'weight'),
    ]
    for s, t, w in preferred:
        if s in colset and t in colset:
            return s, t, w if w in colset else None
    cols = list(columns)
    if len(cols) < 2:
        raise ValueError('Need at least two columns for source/target')
    w = 'weight' if 'weight' in colset else None
    return cols[0], cols[1], w


In [ ]:
# --- config + input path ---
start_dir = Path.cwd().resolve()
project_root = start_dir
config_path = None

for root in [start_dir, *start_dir.parents]:
    candidate = root / 'core_pipeline' / 'configs' / 'config.yaml'
    if candidate.exists():
        project_root = root
        config_path = candidate
        break

if config_path is None:
    raise FileNotFoundError(
        f'config.yaml not found from starting directory: {start_dir}'
    )

cfg = _load_config(config_path)
run_label = str((cfg.get('run') or {}).get('label', '2prat'))

paths_cfg = cfg.get('paths') if isinstance(cfg.get('paths'), dict) else {}
raw_path = paths_cfg.get('edgelist_weighted_lwcc_csv', 'core_pipeline/outputs/runs/{run.label}/01_data/edgelist_weighted_lwcc.csv')
raw_path = str(raw_path).replace('{run.label}', run_label)
edgelist_path = _resolve(project_root, raw_path)

if not edgelist_path.exists():
    raise FileNotFoundError(f'LWCC weighted edgelist not found: {edgelist_path}')

print(f'run_label: {run_label}')
print(f'edgelist_path: {edgelist_path}')

# --- load weighted directed LWCC edgelist ---
header = pd.read_csv(edgelist_path, nrows=0)
src_col, tgt_col, w_col = _detect_cols(header.columns)
usecols = [src_col, tgt_col] + ([w_col] if w_col is not None else [])
df = pd.read_csv(edgelist_path, usecols=usecols, dtype=str)

if w_col is None:
    df['weight'] = 1
else:
    df['weight'] = pd.to_numeric(df[w_col], errors='coerce').fillna(0).astype('int64')

df['source'] = df[src_col].astype(str).str.strip()
df['target'] = df[tgt_col].astype(str).str.strip()
df = df[(df['source'] != '') & (df['target'] != '') & (df['weight'] > 0)].copy()

# ignore self-loops (same convention used in hubness pipeline)
df = df[df['source'] != df['target']].copy()

nodes = pd.Index(sorted(set(df['source']).union(set(df['target']))), dtype='object')
in_strength = df.groupby('target')['weight'].sum().reindex(nodes, fill_value=0).astype('int64')
out_strength = df.groupby('source')['weight'].sum().reindex(nodes, fill_value=0).astype('int64')

stats = pd.DataFrame({
    'node_id': nodes.astype(str),
    'in_strength': in_strength.to_numpy(dtype='int64'),
    'out_strength': out_strength.to_numpy(dtype='int64'),
})

in_s = stats['in_strength'].to_numpy(dtype=float)
out_s = stats['out_strength'].to_numpy(dtype=float)
prob = np.full(len(stats), np.nan, dtype=float)
mask_in_pos = in_s > 0
prob[mask_in_pos] = (in_s[mask_in_pos] - out_s[mask_in_pos]) / in_s[mask_in_pos]
stats['prob_to_sink'] = prob

is_eq1 = np.isclose(stats['prob_to_sink'], 1.0, equal_nan=False)

# 1) all nodes in LWCC
n_all = len(stats)
num_all = int(is_eq1.sum())
frac_all = num_all / n_all if n_all > 0 else np.nan

# 2) nodes with in_strength > 0
mask_in_gt_0 = stats['in_strength'] > 0
n_in_gt_0 = int(mask_in_gt_0.sum())
num_in_gt_0 = int((is_eq1 & mask_in_gt_0.to_numpy()).sum())
frac_in_gt_0 = num_in_gt_0 / n_in_gt_0 if n_in_gt_0 > 0 else np.nan

# 3) nodes with in_strength > out_strength
mask_in_gt_out = stats['in_strength'] > stats['out_strength']
n_in_gt_out = int(mask_in_gt_out.sum())
num_in_gt_out = int((is_eq1 & mask_in_gt_out.to_numpy()).sum())
frac_in_gt_out = num_in_gt_out / n_in_gt_out if n_in_gt_out > 0 else np.nan

print('--- prob_to_sink == 1 fractions ---')
print(f'1) all LWCC nodes:                    {frac_all:.6f}   ({num_all}/{n_all})')
print(f'2) nodes with in_strength > 0:        {frac_in_gt_0:.6f}   ({num_in_gt_0}/{n_in_gt_0})')
print(f'3) nodes with in_strength > out_strength: {frac_in_gt_out:.6f}   ({num_in_gt_out}/{n_in_gt_out})')


## Reversed orientation

This section computes the same `prob_to_sink == 1` fractions after reversing edge direction
(i.e., source/target swapped).


In [ ]:
# --- reversed orientation (swap edge direction) ---
required_cols = {'source', 'target', 'weight'}
if 'df' not in globals() or not required_cols.issubset(set(df.columns)):
    raise RuntimeError('Run the previous data-loading cell first (df with source/target/weight).')

df_rev = df[['target', 'source', 'weight']].rename(columns={'target': 'source', 'source': 'target'}).copy()

nodes_rev = pd.Index(sorted(set(df_rev['source']).union(set(df_rev['target']))), dtype='object')
in_strength_rev = df_rev.groupby('target')['weight'].sum().reindex(nodes_rev, fill_value=0).astype('int64')
out_strength_rev = df_rev.groupby('source')['weight'].sum().reindex(nodes_rev, fill_value=0).astype('int64')

stats_rev = pd.DataFrame({
    'node_id': nodes_rev.astype(str),
    'in_strength': in_strength_rev.to_numpy(dtype='int64'),
    'out_strength': out_strength_rev.to_numpy(dtype='int64'),
})

in_s_rev = stats_rev['in_strength'].to_numpy(dtype=float)
out_s_rev = stats_rev['out_strength'].to_numpy(dtype=float)
prob_rev = np.full(len(stats_rev), np.nan, dtype=float)
mask_in_pos_rev = in_s_rev > 0
prob_rev[mask_in_pos_rev] = (in_s_rev[mask_in_pos_rev] - out_s_rev[mask_in_pos_rev]) / in_s_rev[mask_in_pos_rev]
stats_rev['prob_to_sink'] = prob_rev

is_eq1_rev = np.isclose(stats_rev['prob_to_sink'], 1.0, equal_nan=False)

# 1) all nodes in LWCC
n_all_rev = len(stats_rev)
num_all_rev = int(is_eq1_rev.sum())
frac_all_rev = num_all_rev / n_all_rev if n_all_rev > 0 else np.nan

# 2) nodes with in_strength > 0
mask_in_gt_0_rev = stats_rev['in_strength'] > 0
n_in_gt_0_rev = int(mask_in_gt_0_rev.sum())
num_in_gt_0_rev = int((is_eq1_rev & mask_in_gt_0_rev.to_numpy()).sum())
frac_in_gt_0_rev = num_in_gt_0_rev / n_in_gt_0_rev if n_in_gt_0_rev > 0 else np.nan

# 3) nodes with in_strength > out_strength
mask_in_gt_out_rev = stats_rev['in_strength'] > stats_rev['out_strength']
n_in_gt_out_rev = int(mask_in_gt_out_rev.sum())
num_in_gt_out_rev = int((is_eq1_rev & mask_in_gt_out_rev.to_numpy()).sum())
frac_in_gt_out_rev = num_in_gt_out_rev / n_in_gt_out_rev if n_in_gt_out_rev > 0 else np.nan

print('--- prob_to_sink == 1 fractions (REVERSED) ---')
print(f'1) all LWCC nodes:                    {frac_all_rev:.6f}   ({num_all_rev}/{n_all_rev})')
print(f'2) nodes with in_strength > 0:        {frac_in_gt_0_rev:.6f}   ({num_in_gt_0_rev}/{n_in_gt_0_rev})')
print(f'3) nodes with in_strength > out_strength: {frac_in_gt_out_rev:.6f}   ({num_in_gt_out_rev}/{n_in_gt_out_rev})')

# Optional identity check vs attention (if available from previous cell):
if {'node_id', 'in_strength', 'out_strength'}.issubset(set(globals().get('stats', pd.DataFrame()).columns)):
    att = stats[['node_id', 'in_strength', 'out_strength']].copy()
    chk = att.merge(stats_rev[['node_id', 'in_strength', 'out_strength']], on='node_id', how='inner', suffixes=('_att', '_rev'))
    max_diff_out_in = (chk['out_strength_rev'] - chk['in_strength_att']).abs().max()
    max_diff_in_out = (chk['in_strength_rev'] - chk['out_strength_att']).abs().max()
    print('identity check (rev out == att in): max abs diff =', int(max_diff_out_in))
    print('identity check (rev in  == att out): max abs diff =', int(max_diff_in_out))
